# PyTorch: from tensors to a trained digit classifier

A digit classifier needs a transparent training loop and evidence that its predictions survive saving and reloading. Begin with tensors and a gradient you can verify by hand, then follow the same ideas through batches, loss, validation and held-out errors.

A model learns by adjusting numerical parameters until its predictions fit examples. PyTorch gives us tensors to hold data, automatic differentiation to calculate gradients, and neural-network and optimization tools to train models.

This notebook starts with tensor operations, autograd, and a tiny regression example, then applies those ideas in an end-to-end handwritten-digit classification project. The project uses scikit-learn's bundled 8-by-8 images and produces a trained model, evaluation plots, a metrics report, and test predictions. No data downloads or GPU are required; run the cells from top to bottom.

> **Prerequisites:** Use PyTorch 2.0 or later, NumPy, scikit-learn, and Matplotlib in the notebook's Python environment. Select the PyTorch installation command for your operating system and hardware at [pytorch.org/get-started/locally](https://pytorch.org/get-started/locally/).

## 1. Import PyTorch and select a device

The main package is imported as `torch`. Neural-network layers, datasets, and optimization tools live in `torch.nn`, `torch.utils.data`, and `torch.optim`. We also import scikit-learn utilities for a small built-in image dataset and trustworthy evaluation metrics.

The setup selects CUDA when available, otherwise Apple Metal (MPS), and otherwise the CPU. The tutorial keeps its data small so that CPU execution is practical. A fixed random seed improves repeatability, though hardware and library versions can still cause small differences.

In [ ]:
import copy
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import load_digits
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import train_test_split

torch.manual_seed(42)
np.random.seed(42)

if torch.cuda.is_available():
    device = torch.device('cuda')
elif getattr(torch.backends, 'mps', None) is not None and torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

print('PyTorch version:', torch.__version__)
print('Selected device:', device)

## 2. Create tensors and inspect their shape

A tensor is a container for numerical values. Like a NumPy array, it can have one or more dimensions; unlike a plain array, it can also track calculations for automatic differentiation. Its `shape` describes the layout, and `dtype` describes the kind of values it stores.

| Tensor | Shape | Meaning |
| --- | --- | --- |
| `[2, 4, 6]` | `(3,)` | Three values in one dimension |
| `[[1, 2], [3, 4]]` | `(2, 2)` | Two rows and two columns |

In [ ]:
measurements = torch.tensor([2.0, 4.0, 6.0])
table = torch.tensor([[1, 2], [3, 4]])

print('Measurements:', measurements)
print('Shape:', measurements.shape)
print('Dimensions:', measurements.ndim)
print('Data type:', measurements.dtype)
print('Table shape:', table.shape)

## 3. Transform and combine tensors

Many tensor operations look familiar: indexing selects values, arithmetic applies element by element, and `reshape` changes the layout without changing the data. Broadcasting lets compatible shapes participate in one operation. Here a single offset is added to every item in a two-row table.

In [ ]:
values = torch.tensor([[10.0, 20.0, 30.0], [40.0, 50.0, 60.0]])
offset = torch.tensor([1.0, 2.0, 3.0])

print('First row:', values[0])
print('After broadcasting:', values + offset)
print('Doubled:', values * 2)
print('Reshaped:', values.reshape(3, 2))

## 4. Share data with NumPy and select a device

On the CPU, `torch.from_numpy` and a NumPy array can share the same memory, so changing one may change the other. Use `.clone()` when an independent copy is needed. PyTorch tensors can also move to an accelerator; this example prefers CUDA, then Apple Metal (MPS), and otherwise uses the CPU. The same device should be used for a model and the tensors it processes.

In [ ]:
import numpy as np

array = np.array([1.0, 2.0, 3.0], dtype=np.float32)
shared_tensor = torch.from_numpy(array)
independent_tensor = shared_tensor.clone()
shared_tensor[0] = 9.0

if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

print('NumPy array after shared tensor changed:', array)
print('Independent copy:', independent_tensor)
print('Selected device:', device)
print('Tensor on selected device:', shared_tensor.to(device))

## 5. Verify a gradient before trusting an optimizer

Predict the derivative in the tiny example before backward runs. Compare the computed gradient with your hand calculation. Explain why gradients must be cleared between updates when accumulation is not intended.

Training needs to know how a change in a parameter affects the loss. When `requires_grad=True`, PyTorch records the operations used to produce a result. Calling `.backward()` computes derivatives, which are available on each leaf tensor's `.grad`.

For $y = x^2 + 3x$ at $x = 2$, the derivative is $2x + 3 = 7$.

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = x**2 + 3 * x
y.backward()

print('Value of y:', y.item())
print('Derivative dy/dx:', x.grad.item())

## 6. Train a small regression model

Suppose measurements follow the rule $y = 3x - 0.5$. We will create synthetic input-output pairs, then ask a linear layer to learn the slope and intercept. The training loop repeats four steps: predict, measure error, calculate gradients, and update parameters.

`MSELoss` measures the average squared prediction error. `SGD` changes each parameter in the direction that reduces this loss. A fixed seed makes the initial model parameters repeatable.

In [ ]:
torch.manual_seed(7)
features = torch.linspace(-2, 2, 40).unsqueeze(1)
targets = 3 * features - 0.5

model = nn.Linear(in_features=1, out_features=1)
loss_function = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

with torch.no_grad():
    initial_loss = loss_function(model(features), targets).item()

for epoch in range(150):
    predictions = model(features)
    loss = loss_function(predictions, targets)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

with torch.no_grad():
    final_loss = loss_function(model(features), targets).item()
    learned_slope = model.weight.item()
    learned_intercept = model.bias.item()

print(f'Initial loss: {initial_loss:.4f}')
print(f'Final loss:   {final_loss:.6f}')
print(f'Learned rule: y = {learned_slope:.3f}x + {learned_intercept:.3f}')

The loss should become much smaller, and the learned slope and intercept should be close to `3` and `-0.5`. `zero_grad()` matters because PyTorch accumulates gradients by default; clearing them starts each training step with a clean gradient. `torch.no_grad()` avoids recording operations when we only need predictions for inspection.

## 7. Load examples in batches

A `Dataset` represents examples and their labels, while a `DataLoader` groups those examples into batches and can shuffle their order. `TensorDataset` is a convenient dataset for tensors that share the same first dimension.

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

dataset = TensorDataset(features, targets)
loader = DataLoader(dataset, batch_size=8, shuffle=True, generator=torch.Generator().manual_seed(7))

batch_features, batch_targets = next(iter(loader))
print('Feature batch shape:', batch_features.shape)
print('Target batch shape:', batch_targets.shape)
print('Number of batches:', len(loader))

## 8. Save and restore learned parameters

A model's `state_dict` contains its learned parameters. Saving that dictionary is a lightweight way to preserve training results. To restore it, create the same model structure and load the saved state. `map_location` lets a checkpoint load on a CPU even if it was saved on another device.

In [ ]:
checkpoint_path = 'linear_model.pth'
torch.save(model.state_dict(), checkpoint_path)

restored_model = nn.Linear(in_features=1, out_features=1)
state = torch.load(checkpoint_path, map_location='cpu', weights_only=True)
restored_model.load_state_dict(state)
restored_model.eval()

with torch.no_grad():
    sample_prediction = restored_model(torch.tensor([[2.0]]))
print('Prediction for x=2:', sample_prediction.item())

## Project: handwritten digit recognition

We will turn the tensor and training-loop foundations above into a complete image-classification workflow. The goal is to recognize digits from small 8-by-8 grayscale images using scikit-learn's built-in `load_digits` dataset, so the project needs no downloads.

The project will define an input contract, keep training, validation, and test data separate, compare against a simple majority-class baseline, train a neural network with early stopping, inspect mistakes, and save a model plus a reproducible report. The final test set is used only after model choices are complete. This is a compact benchmark project, not a claim of accuracy on real-world handwriting.

## 9. Define what counts as a valid digit input

Inspect image size, pixel range and label encoding before building the network. Predict the shape after flattening. A tensor with the right number of elements can still be wrong if its pixels are ordered or scaled differently.

The scikit-learn digits dataset contains 1,797 labeled grayscale images. Each image is 8 by 8 pixels; original pixel intensities range from 0 to 16, and labels are integers from 0 through 9. We will convert images to `float32`, divide by 16 so values lie in `[0, 1]`, and use the PyTorch channel-first layout `(batch, channels, height, width)`.

We reserve 20% for a final test, then split the remaining examples into 75% training and 25% validation. This yields a 60/20/20 split overall. Stratification preserves class proportions. The majority-class accuracy is our simple baseline: a useful model should beat it by a substantial margin.

In [ ]:
digits = load_digits()
raw_images = digits.images.astype(np.float32)
labels = digits.target.astype(np.int64)
features = raw_images[:, np.newaxis, :, :] / 16.0

all_indices = np.arange(len(labels))
train_val_indices, test_indices = train_test_split(
    all_indices, test_size=0.2, stratify=labels, random_state=42
)
train_indices, validation_indices = train_test_split(
    train_val_indices,
    test_size=0.25,
    stratify=labels[train_val_indices],
    random_state=42,
)

x_train, y_train = features[train_indices], labels[train_indices]
x_validation, y_validation = features[validation_indices], labels[validation_indices]
x_test, y_test = features[test_indices], labels[test_indices]
majority_class = int(np.bincount(y_train, minlength=10).argmax())
majority_baseline = float(np.mean(y_test == majority_class))

print('Full dataset:', raw_images.shape, 'images')
print('Input tensor shape:', features.shape, 'pixel range:', features.min(), features.max())
print('Split sizes:', len(train_indices), len(validation_indices), len(test_indices))
print('Majority-class baseline accuracy:', round(majority_baseline, 3))

## 10. Inspect image examples before modeling

Plotting a few raw images verifies the orientation, intensity range, and relationship between each image and its label. This simple inspection catches common data bugs before training. The model input will be the normalized channel-first tensor; the plot uses the original 8-by-8 image for readability.

In [ ]:
figure, axes = plt.subplots(2, 5, figsize=(10, 4), layout='constrained')
for axis, image, label in zip(axes.flat, raw_images[:10], labels[:10]):
    axis.imshow(image, cmap='gray', vmin=0, vmax=16)
    axis.set_title(f'Label {label}')
    axis.axis('off')
plt.show()

## 11. Batch the data and prepare the device

A `TensorDataset` keeps each image paired with its label. The `DataLoader` groups examples into mini-batches; shuffling the training examples each epoch reduces dependence on their original order. Validation and test loaders stay unshuffled so their predictions align with their source arrays.

Data stays in host memory until each batch is sent to the selected device. We use zero worker processes to keep notebook execution simple and portable.

In [ ]:
train_dataset = TensorDataset(torch.from_numpy(x_train), torch.from_numpy(y_train))
validation_dataset = TensorDataset(torch.from_numpy(x_validation), torch.from_numpy(y_validation))
test_dataset = TensorDataset(torch.from_numpy(x_test), torch.from_numpy(y_test))

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    generator=torch.Generator().manual_seed(42),
    pin_memory=device.type == 'cuda',
)
validation_loader = DataLoader(validation_dataset, batch_size=128, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

image_batch, label_batch = next(iter(train_loader))
print('Batch image shape:', image_batch.shape)
print('Batch label shape:', label_batch.shape)
print('Batch dtype:', image_batch.dtype, label_batch.dtype)
print('Training batches per epoch:', len(train_loader))

## 12. Build the neural network

This is a small multilayer perceptron (MLP). `Flatten` turns each 1-by-8-by-8 image into 64 values; a hidden layer learns combinations of pixels, ReLU adds nonlinearity, and dropout regularizes training. The final layer produces ten **logits**, one for each digit. We do not put softmax in the model because `CrossEntropyLoss` consumes logits directly and applies the needed normalization internally.

In [ ]:
class DigitClassifier(nn.Module):
    def __init__(self, hidden_units=128, dropout_rate=0.2):
        super().__init__()
        self.network = nn.Sequential(
            nn.Flatten(),
            nn.Linear(8 * 8, hidden_units),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_units, 10),
        )

    def forward(self, images):
        return self.network(images)

model = DigitClassifier().to(device)
parameter_count = sum(parameter.numel() for parameter in model.parameters())
print(model)
print(f'Trainable parameters: {parameter_count:,}')

## 13. Define the loss and evaluation helper

For multi-class classification with integer labels, cross-entropy is the standard starting loss. Adam adapts parameter updates using gradients. The evaluation helper switches the model to inference mode, disables gradient tracking, and aggregates loss and predictions across all batches; it does not update weights.

In [ ]:
loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)


def evaluate(model, data_loader):
    model.eval()
    total_loss = 0.0
    all_predictions = []
    all_targets = []
    all_probabilities = []

    with torch.inference_mode():
        for batch_images, batch_labels in data_loader:
            batch_images = batch_images.to(device, non_blocking=device.type == 'cuda')
            batch_labels = batch_labels.to(device, non_blocking=device.type == 'cuda')
            logits = model(batch_images)
            loss = loss_function(logits, batch_labels)
            total_loss += loss.item() * len(batch_labels)
            all_predictions.append(logits.argmax(dim=1).cpu().numpy())
            all_targets.append(batch_labels.cpu().numpy())
            all_probabilities.append(torch.softmax(logits, dim=1).cpu().numpy())

    average_loss = total_loss / len(data_loader.dataset)
    return (
        average_loss,
        np.concatenate(all_predictions),
        np.concatenate(all_targets),
        np.concatenate(all_probabilities),
    )

print('Loss:', loss_function.__class__.__name__)
print('Optimizer:', optimizer.__class__.__name__)

## 14. Let validation guide stopping without training on it

Predict what would suggest overfitting in the learning curves. Identify where training and evaluation modes change and where gradients are disabled. Confirm that the saved best weights are selected by validation rather than test performance.

Each epoch passes over the training batches. The training loop clears old gradients, computes logits and loss, calls `backward()` to calculate gradients, and asks the optimizer to update parameters. It then measures validation loss and accuracy without updating weights.

We keep a copy of the weights with the lowest validation loss and stop after six epochs without improvement. This is a practical guard against overfitting and wasted work; the validation set guides training, while the test set remains untouched.

In [ ]:
history = {'train_loss': [], 'validation_loss': [], 'validation_accuracy': []}
best_validation_loss = float('inf')
best_state = None
best_epoch = 0
patience = 6
epochs_without_improvement = 0
maximum_epochs = 40

for epoch in range(1, maximum_epochs + 1):
    model.train()
    total_training_loss = 0.0

    for batch_images, batch_labels in train_loader:
        batch_images = batch_images.to(device, non_blocking=device.type == 'cuda')
        batch_labels = batch_labels.to(device, non_blocking=device.type == 'cuda')

        optimizer.zero_grad(set_to_none=True)
        logits = model(batch_images)
        loss = loss_function(logits, batch_labels)
        loss.backward()
        optimizer.step()
        total_training_loss += loss.item() * len(batch_labels)

    training_loss = total_training_loss / len(train_loader.dataset)
    validation_loss, validation_predictions, validation_targets, _ = evaluate(model, validation_loader)
    validation_accuracy = accuracy_score(validation_targets, validation_predictions)
    history['train_loss'].append(training_loss)
    history['validation_loss'].append(validation_loss)
    history['validation_accuracy'].append(validation_accuracy)

    if validation_loss < best_validation_loss - 1e-4:
        best_validation_loss = validation_loss
        best_state = copy.deepcopy(model.state_dict())
        best_epoch = epoch
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epoch == 1 or epoch % 5 == 0:
        print(
            f'Epoch {epoch:02d}: train loss={training_loss:.4f}, '
            f'validation loss={validation_loss:.4f}, '
            f'validation accuracy={validation_accuracy:.3f}'
        )

    if epochs_without_improvement >= patience:
        print(f'Early stopping after epoch {epoch}.')
        break

model.load_state_dict(best_state)
print(f'Best validation-loss epoch: {best_epoch}')

## 15. Read the learning curves

The training and validation losses help diagnose the optimization process. If training loss keeps falling while validation loss rises, the model may be overfitting. If both remain high, the model may be underfitting or the data and optimization setup may need attention. The vertical marker identifies the checkpoint restored by early stopping.

In [ ]:
epoch_numbers = np.arange(1, len(history['train_loss']) + 1)
learning_figure, axes = plt.subplots(1, 2, figsize=(10, 4), layout='constrained')
axes[0].plot(epoch_numbers, history['train_loss'], label='Training')
axes[0].plot(epoch_numbers, history['validation_loss'], label='Validation')
axes[0].axvline(best_epoch, color='gray', linestyle='--', label='Best checkpoint')
axes[0].set(xlabel='Epoch', ylabel='Cross-entropy loss', title='Loss')
axes[0].legend()
axes[1].plot(epoch_numbers, history['validation_accuracy'], color='tab:green')
axes[1].axvline(best_epoch, color='gray', linestyle='--', label='Best checkpoint')
axes[1].set(xlabel='Epoch', ylabel='Accuracy', title='Validation accuracy')
axes[1].legend()
plt.show()

## 16. Evaluate once on the held-out test set

After selecting the checkpoint with validation data, measure the final model on the test set. Compare accuracy with the majority baseline, and include macro F1 and balanced accuracy so performance across classes is visible. The confusion matrix shows which digits are most often confused. These results estimate performance on this benchmark split; they do not guarantee performance on different handwriting or image capture conditions.

In [ ]:
test_loss, test_predictions, test_targets, test_probabilities = evaluate(model, test_loader)
test_accuracy = accuracy_score(test_targets, test_predictions)
test_macro_f1 = f1_score(test_targets, test_predictions, average='macro')
test_balanced_accuracy = balanced_accuracy_score(test_targets, test_predictions)

print(f'Test loss: {test_loss:.4f}')
print(f'Test accuracy: {test_accuracy:.3f} (majority baseline: {majority_baseline:.3f})')
print(f'Test macro F1: {test_macro_f1:.3f}')
print(f'Test balanced accuracy: {test_balanced_accuracy:.3f}')
print()
print(classification_report(
    test_targets,
    test_predictions,
    labels=list(range(10)),
    target_names=[str(label) for label in digits.target_names],
    zero_division=0,
))

confusion_figure, axis = plt.subplots(figsize=(7, 6), layout='constrained')
ConfusionMatrixDisplay(
    confusion_matrix=confusion_matrix(test_targets, test_predictions),
    display_labels=digits.target_names,
).plot(cmap='Blues', colorbar=False, ax=axis)
axis.set_title('Held-out digit predictions')
plt.show()

## 17. Use mistakes to describe the model’s boundary

Inspect true and predicted labels for several errors. Separate visual ambiguity from preprocessing mistakes. These examples characterize the fixed evaluation; repeatedly redesigning the model around them would spend the test set.

Overall scores hide individual failures. Display a handful of incorrect test predictions with their true labels and predicted probabilities. If there are fewer than eight errors, the remaining panels show correct predictions for context. Use this analysis to form hypotheses for future improvements, not to repeatedly tune against the test set.

In [ ]:
incorrect_indices = np.flatnonzero(test_predictions != test_targets)
if len(incorrect_indices) >= 8:
    preview_indices = incorrect_indices[:8]
else:
    preview_indices = np.arange(min(8, len(test_targets)))

error_figure, axes = plt.subplots(2, 4, figsize=(10, 5), layout='constrained')
for axis, sample_index in zip(axes.flat, preview_indices):
    predicted_label = int(test_predictions[sample_index])
    confidence = float(test_probabilities[sample_index, predicted_label])
    axis.imshow(raw_images[test_indices[sample_index]], cmap='gray', vmin=0, vmax=16)
    axis.set_title(
        f'True {test_targets[sample_index]}, predicted {predicted_label}\n'
        f'Predicted-class probability {confidence:.2f}'
    )
    axis.axis('off')
plt.show()
print('Misclassified test images:', len(incorrect_indices))

## 18. Save, reload, and predict

A `state_dict` stores learned parameter tensors, while the architecture is defined by Python code. Save a small configuration and input contract next to the weights so that a future inference program knows how to reconstruct the model and prepare inputs. We reload with `weights_only=True`, map tensors to the current device, and compare predictions to verify the round trip.

In [ ]:
output_dir = Path.cwd() / 'outputs'
output_dir.mkdir(exist_ok=True)
model_path = output_dir / 'pytorch_digit_classifier.pth'
model_config = {'hidden_units': 128, 'dropout_rate': 0.2}
input_contract = {
    'shape_without_batch': [1, 8, 8],
    'dtype': 'float32',
    'pixel_range': [0.0, 1.0],
    'normalization': 'original integer pixel values divided by 16',
    'class_labels': list(range(10)),
}
checkpoint = {
    'model_config': model_config,
    'input_contract': input_contract,
    'model_state_dict': model.state_dict(),
}
torch.save(checkpoint, model_path)

loaded_checkpoint = torch.load(model_path, map_location=device, weights_only=True)
restored_model = DigitClassifier(**loaded_checkpoint['model_config']).to(device)
restored_model.load_state_dict(loaded_checkpoint['model_state_dict'])
_, restored_predictions, _, _ = evaluate(restored_model, test_loader)
print('Predictions match after reload:', np.array_equal(test_predictions, restored_predictions))
print('Input contract:', loaded_checkpoint['input_contract'])

sample_tensor = torch.from_numpy(x_test[:1]).to(device)
restored_model.eval()
with torch.inference_mode():
    sample_probabilities = torch.softmax(restored_model(sample_tensor), dim=1).cpu().numpy()[0]
print('Sample predicted digit:', int(np.argmax(sample_probabilities)))
print('Sample probability distribution:', np.round(sample_probabilities, 3))

## 19. Export project artifacts

Save the learning curves, confusion matrix, test predictions, and a compact JSON report beside the model. Keeping the input contract, split sizes, package version, and metrics together makes the result easier to reproduce and review. These files are generated under `outputs/`; do not treat benchmark artifacts as production monitoring or as a substitute for tracking data and code versions.

In [ ]:
learning_figure.savefig(output_dir / 'pytorch_learning_curves.png', dpi=150)
confusion_figure.savefig(output_dir / 'pytorch_confusion_matrix.png', dpi=150)
project_report = {
    'pytorch_version': str(torch.__version__),
    'device': str(device),
    'split_sizes': {
        'training': int(len(train_indices)),
        'validation': int(len(validation_indices)),
        'test': int(len(test_indices)),
    },
    'best_epoch': int(best_epoch),
    'majority_baseline_accuracy': float(majority_baseline),
    'test_metrics': {
        'loss': float(test_loss),
        'accuracy': float(test_accuracy),
        'macro_f1': float(test_macro_f1),
        'balanced_accuracy': float(test_balanced_accuracy),
    },
    'input_contract': input_contract,
    'history': {key: [float(value) for value in values] for key, values in history.items()},
}
(output_dir / 'pytorch_digit_report.json').write_text(
    json.dumps(project_report, indent=2) + '\n', encoding='utf-8'
)
np.savez_compressed(
    output_dir / 'pytorch_digit_predictions.npz',
    source_indices=test_indices,
    actual=test_targets,
    predicted=test_predictions,
    probabilities=test_probabilities,
)
print('Saved model, two figures, JSON report, and compressed test predictions.')

## Project recap and next experiments

This project moves from raw 8-by-8 pixels to a saved digit classifier. It uses stratified data splits, an explicit normalization and shape contract, mini-batch optimization, validation-based early stopping, a final test evaluation, and a verified save/reload path.

Useful extensions:

1. Replace the MLP with a small convolutional network and compare validation performance.
2. Tune hidden size, dropout, and learning rate using validation data only.
3. Inspect errors by class and test whether a confusion-aware augmentation or sampling strategy helps.
4. Evaluate probability calibration if downstream decisions depend on confidence.
5. For real handwriting, collect representative data and define a fresh test set; this tiny benchmark is not a production guarantee.

Keep the test set out of repeated model selection. Record code, dependency versions, and preprocessing alongside any model used beyond this notebook.

## Explain what the evidence supports

Explain one incorrect prediction with reference to the image and model input contract. Then verify that a reloaded model produces matching predictions on the same fixed inputs.

Change one assumption or input at a time. Predict the effect before rerunning, compare it with the actual result, and explain any difference. Keep the original result available so that the comparison is meaningful.

## Second project: Keep digit recognition useful when scans deteriorate

Use scikit-learn’s bundled handwritten digits, then inject explicitly simulated duplicate deliveries, absent labels, missing pixels and scanner noise. Compare ordinary training with noise augmentation under the same eight-epoch CPU budget.

Run these cells in order. This extension uses its own `p2_` variables and writes only its named outputs under `outputs/project2/`. Predict each comparison before running it; the worked discussion after the code explains the result and its limits.

### Work through the data boundary

The pixel scale is 0–16, so dividing once by 16 produces 0–1 inputs. Dividing again would compress the signal to 0–0.0625: a shape assertion would miss that bug. Split original image IDs before adding duplicates; otherwise the same handwriting can appear in training and evaluation. Remove exact repeated deliveries, quarantine five unlabeled training rows, and estimate missing-pixel medians only from training. A missing label cannot be reconstructed from its row number.

Validation and test noise are generated once and retained. This makes comparisons paired: both models see the same difficult examples. Noise is a controlled stress test, not evidence about a particular real scanner.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import time
from IPython.display import display
p2_out=Path('outputs/project2');p2_out.mkdir(parents=True,exist_ok=True)

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
p2_digits=load_digits();p2_ids=np.arange(len(p2_digits.target))
p2_dev,p2_test_ids=train_test_split(p2_ids,test_size=.2,stratify=p2_digits.target,random_state=71)
p2_train_ids,p2_val_ids=train_test_split(p2_dev,test_size=.2,stratify=p2_digits.target[p2_dev],random_state=71)
p2_rng=np.random.default_rng(71)
p2_raw=pd.DataFrame(p2_digits.data[p2_train_ids],columns=[f'pixel_{i}' for i in range(64)])
p2_raw['id']=p2_train_ids;p2_raw['label']=p2_digits.target[p2_train_ids]
p2_raw.loc[:4,'label']=np.nan
p2_raw=pd.concat([p2_raw,p2_raw.iloc[-12:]],ignore_index=True)
p2_clean=p2_raw.drop_duplicates().dropna(subset=['label']).copy()
assert p2_clean.id.is_unique
p2_x=p2_clean.iloc[:,:64].to_numpy(dtype=np.float32)/16
p2_y=p2_clean.label.to_numpy(dtype=np.int64)
p2_missing=p2_rng.random(p2_x.shape)<.04;p2_x[p2_missing]=np.nan
p2_medians=np.nanmedian(p2_x,axis=0)
p2_x=np.where(np.isnan(p2_x),p2_medians,p2_x).astype('float32')
p2_v=p2_digits.data[p2_val_ids].astype('float32')/16
p2_t=p2_digits.data[p2_test_ids].astype('float32')/16
p2_vy=p2_digits.target[p2_val_ids];p2_ty=p2_digits.target[p2_test_ids]
def p2_corrupt(x,rng):
    return np.clip(x+rng.normal(0,.22,x.shape),0,1).astype('float32')
p2_v_bad=p2_corrupt(p2_v,p2_rng);p2_t_bad=p2_corrupt(p2_t,p2_rng)
print({'raw_train_rows':len(p2_raw),'accepted':len(p2_x),'validation':len(p2_v),'test':len(p2_t),'imputed_pixels':int(p2_missing.sum())})
assert np.isfinite(p2_x).all() and p2_x.max()<=1

### Spend the training budget deliberately

The 64→32→10 network has 2,410 parameters: 64×32+32 biases, plus 32×10+10 biases. Both approaches use the same initial seed, architecture, optimizer and epoch limit. The augmented approach draws new bounded noise for each training batch. Select each model's checkpoint by noisy validation cross-entropy, then report both clean and noisy test accuracy once. This explicitly favors robustness during selection; a different operational objective could make a different checkpoint appropriate.

In [ ]:
import torch,copy
torch.set_num_threads(2)
def p2_fit(augment):
    torch.manual_seed(71);rng=np.random.default_rng(71)
    net=torch.nn.Sequential(torch.nn.Linear(64,32),torch.nn.ReLU(),torch.nn.Linear(32,10))
    opt=torch.optim.Adam(net.parameters(),lr=.01);best=float('inf');history=[];started=time.perf_counter()
    for epoch in range(8):
        net.train()
        order=rng.permutation(len(p2_x))
        for start in range(0,len(order),64):
            ix=order[start:start+64];x=p2_corrupt(p2_x[ix],rng) if augment else p2_x[ix]
            opt.zero_grad();loss=torch.nn.functional.cross_entropy(net(torch.from_numpy(x)),torch.from_numpy(p2_y[ix]));loss.backward();opt.step()
        net.eval()
        with torch.no_grad(): val=float(torch.nn.functional.cross_entropy(net(torch.from_numpy(p2_v_bad)),torch.from_numpy(p2_vy)))
        history.append(val)
        if val<best:best=val;state=copy.deepcopy(net.state_dict())
    net.load_state_dict(state);net.eval()
    def predict(x):
        with torch.no_grad():return net(torch.from_numpy(x)).argmax(1).numpy()
    return net,predict,history,time.perf_counter()-started

p2_results=[]
for p2_aug in [False,True]:
    p2_net,p2_predict,p2_history,p2_seconds=p2_fit(p2_aug)
    p2_full=p2_predict(p2_t_bad)
    p2_chunk=np.concatenate([p2_predict(p2_t_bad[i:i+32]) for i in range(0,len(p2_t_bad),32)])
    assert np.array_equal(p2_full,p2_chunk)
    p2_results.append({'augmentation':p2_aug,'best_noisy_validation_loss':min(p2_history),'selected_epoch':int(np.argmin(p2_history))+1,'clean_test_accuracy':accuracy_score(p2_ty,p2_predict(p2_t)),'noisy_test_accuracy':accuracy_score(p2_ty,p2_full),'training_seconds':p2_seconds})
display(pd.DataFrame(p2_results))
print({'full_input_bytes':p2_t_bad.nbytes,'batch_32_input_bytes':p2_t_bad[:32].nbytes,'epochs_per_approach':8})

### Interpret the tradeoff and debug the next failure

Augmentation can improve noisy accuracy while sacrificing clean accuracy; it can also fail under this small budget. Report what the table actually shows. Cross-entropy can improve without changing accuracy because confidence changes before the winning class does. The batch check proves identical class decisions here; its byte counts describe input arrays, not total framework memory or production latency.

**Try:** replace Gaussian noise with an occluded 2×2 patch. **Worked reasoning:** additive-noise training does not guarantee occlusion robustness. Define a fresh validation corruption policy before tuning, retain an untouched final test set, and inspect errors by digit. A useful next competing approach is a small convolutional model, whose locality bias differs from this dense network.